# IBKR News Fetch Smoke Test (`ib_async`)

这个 notebook 只测试 `news_api` 的抓新闻链路：`ib_async` 连接 IB、读取 news providers、补拉历史新闻、订阅实时 `tickNews`，并检查 SQLite 原始新闻入库。它不会测试组合监控、期权、Bark 推送或其他项目功能。

## 1. 参数和依赖

运行前确认 TWS 或 IB Gateway 已开启 API。常见端口：实盘 Gateway `4001`，纸账户 Gateway `4002`，TWS 常见 `7496/7497`。

In [ ]:
from pathlib import Path
import logging
import sys
import time
from dataclasses import replace

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "news_api").exists() and (PROJECT_ROOT.parent / "news_api").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

try:
    from ib_async import Contract, IB, Stock, util
except ModuleNotFoundError as exc:
    raise RuntimeError("当前 notebook kernel 没有安装 ib_async，请先安装/切换到本项目运行 IB 的环境。") from exc

try:
    from ib_async.ib import StartupFetch
except Exception:
    StartupFetch = None

from news_api.cleaner import clean_headline, parse_headline_metadata, parse_ib_time
from news_api.config import SETTINGS
from news_api.models import NewsHeadline
from news_api.service import NewsService
from news_api.storage import SQLiteNewsStorage
from news_api.watchlist import normalize_watchlist

# ib_async 自带 logger 会用 repr 打中文，容易出现 \uXXXX；这里关闭它，保留下面自定义 errorEvent 输出。
for logger_name in ("ib_async", "ib_async.client", "ib_async.wrapper"):
    logging.getLogger(logger_name).setLevel(logging.CRITICAL)

# 按你的账户环境调整。
HOST = SETTINGS.host
PORT = SETTINGS.port
CLIENT_ID = 191

# 单票测试只用于验证 historicalNews 和 contract news；全市场新闻流看 BroadTape 测试。
SYMBOLS_TO_TEST = ["ORCL"]
HISTORICAL_RESULTS = 30
HISTORICAL_WAIT_SECONDS = 30
RUN_SINGLE_STOCK_REALTIME = False
REALTIME_WAIT_SECONDS = 120

# 全市场新闻流 smoke test：使用 NEWS 合约订阅已授权 provider 的 broad tape。
BROADTAPE_SYMBOL = "ALL_NEWS"
BROADTAPE_PROVIDER_LIMIT = None  # None = 尝试 reqNewsProviders() 返回的全部 provider
BROADTAPE_WAIT_SECONDS = 180

PROVIDER_CODES = SETTINGS.provider_codes
RUN_ID = time.strftime("%Y%m%d_%H%M%S")
DB_PATH = PROJECT_ROOT / "news_api" / "data" / f"news_fetch_smoke_test_{RUN_ID}.sqlite"

base_watchlist = normalize_watchlist()
WATCHLIST = {
    symbol: base_watchlist[symbol]
    for symbol in SYMBOLS_TO_TEST
    if symbol in base_watchlist
}
missing = sorted(set(SYMBOLS_TO_TEST) - set(WATCHLIST))
if missing:
    raise ValueError(f"这些 symbol 不在 news_api/watchlist.py 中：{missing}")
print("project:", PROJECT_ROOT)
print("db:", DB_PATH)
print("host/port/client_id:", HOST, PORT, CLIENT_ID)
print("provider_codes:", PROVIDER_CODES)
print("single-symbol watchlist:", WATCHLIST)
print("broad tape symbol:", BROADTAPE_SYMBOL)

## 2. 创建只用于抓新闻测试的 service

这里使用单独 SQLite 文件，并把正文抓取和推送阈值调高，避免 smoke test 触发 Bark 或做过多后处理；本 notebook 关注的是 IB 新闻标题是否能正确进来。

In [ ]:
class DryRunBarkClient:
    def push(self, analysis, priority):
        return "skipped", "news_fetch_smoke_test dry run"


test_settings = replace(
    SETTINGS,
    db_path=DB_PATH,
    article_fetch_score=999,
    default_push_score=999,
    portfolio_push_score=999,
)

storage = SQLiteNewsStorage(DB_PATH)
SERVICE_WATCHLIST = {
    **WATCHLIST,
    BROADTAPE_SYMBOL: {
        "exchange": "NEWS",
        "priority": 1,
        "aliases": [BROADTAPE_SYMBOL, "BroadTape", "All News"],
    },
}
service = NewsService(
    settings=test_settings,
    watchlist=SERVICE_WATCHLIST,
    storage=storage,
    bark_client=DryRunBarkClient(),
)
service.start()

def rows(sql, params=()):
    with storage.connect() as connection:
        return [dict(row) for row in connection.execute(sql, params).fetchall()]

def scalar(sql, params=()):
    with storage.connect() as connection:
        return connection.execute(sql, params).fetchone()[0]

print("service ready")

## 3. 连接 IB (`ib_async`)

这里沿用本项目其他模块的 `ib_async` 风格。连接失败时，先查 TWS/Gateway API 设置、端口和 client id 冲突。

In [ ]:
util.startLoop()
ib = IB()
connect_kwargs = {
    "clientId": CLIENT_ID,
    "timeout": 20,
    "readonly": True,
}
if StartupFetch is not None:
    connect_kwargs["fetchFields"] = StartupFetch(0)

def readable_ib_text(value):
    text = str(value)
    if "\\u" not in text and "\\x" not in text:
        return text
    try:
        return text.encode("utf-8").decode("unicode_escape")
    except UnicodeDecodeError:
        return text


errors = []
def on_error(req_id, error_code, error_string, contract):
    message = readable_ib_text(error_string)
    row = {
        "time": time.strftime("%H:%M:%S"),
        "reqId": req_id,
        "errorCode": error_code,
        "errorString": message,
        "contract": readable_ib_text(contract) if contract else "",
    }
    errors.append(row)
    print(f"IBKR message [{row['time']}] reqId={req_id} code={error_code}: {message}")

ib.errorEvent += on_error
ib.connect(HOST, PORT, **connect_kwargs)
print("connected:", ib.isConnected())

## 4. 读取 news providers

`reqNewsProviders()` 能返回 provider 列表，说明 API 侧新闻 provider 查询至少能跑通。

In [ ]:
providers = ib.reqNewsProviders()
provider_rows = [
    {
        "code": getattr(provider, "code", ""),
        "name": getattr(provider, "name", ""),
    }
    for provider in providers
]
print("provider count:", len(provider_rows))
try:
    import pandas as pd
    display(pd.DataFrame(provider_rows))
except ModuleNotFoundError:
    provider_rows

## 5. 合约工具函数

历史新闻需要 `conId`，所以先用 `ib.qualifyContracts()` 确认合约。

In [ ]:
def make_stock_contract(symbol, item):
    primary_exchange = str(item.get("exchange", "SMART") or "SMART")
    kwargs = {}
    if primary_exchange.upper() != "SMART":
        kwargs["primaryExchange"] = primary_exchange
    return Stock(symbol, "SMART", "USD", **kwargs)


contracts = {}
for symbol, item in WATCHLIST.items():
    contract = make_stock_contract(symbol, item)
    qualified = ib.qualifyContracts(contract)
    if not qualified:
        raise RuntimeError(f"无法 qualify 合约：{symbol} {contract}")
    contracts[symbol] = qualified[0]
    print(symbol, "conId=", contracts[symbol].conId, contracts[symbol])

## 6. 历史新闻补拉测试

这一步用 `ib.reqHistoricalNews()` 验证新闻权限、provider code、合约解析和入库。历史新闻能进库，说明抓新闻主链路是通的。

In [ ]:
def ingest_historical_item(symbol, contract, item):
    headline_raw = getattr(item, "headline", "") or ""
    parsed = parse_headline_metadata(headline_raw)
    published_at, _local_time = parse_ib_time(str(getattr(item, "time", "") or ""))
    event = NewsHeadline(
        symbol=symbol,
        provider=str(getattr(item, "providerCode", "") or ""),
        article_id=str(getattr(item, "articleId", "") or ""),
        headline=clean_headline(headline_raw),
        headline_raw=headline_raw,
        publisher=parsed["publisher"],
        published_at=published_at,
        con_id=int(contract.conId or 0),
    )
    return service.ingest_headline(event)


before_raw = scalar("SELECT COUNT(*) FROM news_raw")
historical_counts = {}

for symbol, contract in contracts.items():
    print("request historical news:", symbol, "conId=", contract.conId)
    try:
        items = ib.reqHistoricalNews(
            contract.conId,
            PROVIDER_CODES,
            "",
            "",
            HISTORICAL_RESULTS,
        )
    except TypeError:
        items = ib.reqHistoricalNews(
            conId=contract.conId,
            providerCodes=PROVIDER_CODES,
            startDateTime="",
            endDateTime="",
            totalResults=HISTORICAL_RESULTS,
        )
    historical_counts[symbol] = len(items)
    inserted = 0
    for item in items:
        inserted += int(ingest_historical_item(symbol, contract, item))
    print(symbol, "returned=", len(items), "inserted=", inserted)

service.queue.join()
after_raw = scalar("SELECT COUNT(*) FROM news_raw")
print("historical rows added:", after_raw - before_raw)
if after_raw == before_raw:
    print("没有历史新闻入库。请查看上方 IBKR error，重点检查新闻权限、provider_codes、端口和账户类型。")

In [ ]:
latest_raw = rows(
    """
    SELECT symbol, published_at, provider, article_id, publisher, headline
    FROM news_raw
    ORDER BY received_at DESC
    LIMIT 20
    """
)

try:
    import pandas as pd
    display(pd.DataFrame(latest_raw))
except ModuleNotFoundError:
    latest_raw

## 7. 实时 tickNews 订阅测试

这一步用 `ib.reqMktData(..., genericTickList='mdoff,292:...')` 订阅实时新闻。实时新闻不是每分钟都有；如果历史补拉成功但这里暂时没有新增，通常不是代码错误。

In [ ]:
def iter_news_ticks(ticker):
    for attr in ("newsTicks", "news", "tickNews"):
        value = getattr(ticker, attr, None)
        if isinstance(value, list):
            yield from value


def ingest_realtime_tick(symbol, contract, tick):
    provider = str(getattr(tick, "providerCode", getattr(tick, "provider", "")) or "")
    article_id = str(getattr(tick, "articleId", "") or "")
    headline = str(getattr(tick, "headline", "") or "")
    timestamp = str(getattr(tick, "timeStamp", getattr(tick, "time", "")) or "")
    extra_data = str(getattr(tick, "extraData", "") or "")
    if not provider or not article_id or not headline:
        return False
    return service.ingest_tick_news(
        symbol=symbol,
        timestamp=timestamp,
        provider=provider,
        article_id=article_id,
        headline=headline,
        ticker_id=None,
        extra_data=extra_data,
    )


STOCK_NEWS_GENERIC_TICKS = f"mdoff,292:{PROVIDER_CODES}"
NEWS_GENERIC_TICKS = "mdoff,292"
tickers = {}
seen = set()
before_realtime = scalar("SELECT COUNT(*) FROM news_raw")

if not RUN_SINGLE_STOCK_REALTIME:
    print("skip single-stock realtime news test; set RUN_SINGLE_STOCK_REALTIME=True to run it")
else:
    for symbol, contract in contracts.items():
        tickers[symbol] = ib.reqMktData(contract, genericTickList=STOCK_NEWS_GENERIC_TICKS, snapshot=False, regulatorySnapshot=False)
        print("subscribed:", symbol, STOCK_NEWS_GENERIC_TICKS)

    deadline = time.time() + REALTIME_WAIT_SECONDS
    while time.time() < deadline:
        ib.sleep(5)
        inserted = 0
        observed = 0
        for symbol, ticker in tickers.items():
            contract = contracts[symbol]
            for tick in iter_news_ticks(ticker):
                provider = str(getattr(tick, "providerCode", getattr(tick, "provider", "")) or "")
                article_id = str(getattr(tick, "articleId", "") or "")
                key = (symbol, provider, article_id)
                if key in seen:
                    continue
                seen.add(key)
                observed += 1
                inserted += int(ingest_realtime_tick(symbol, contract, tick))
        service.queue.join()
        current = scalar("SELECT COUNT(*) FROM news_raw")
        print(f"raw rows: {current} (+{current - before_realtime} realtime), observed={observed}, inserted={inserted}")
        if current > before_realtime:
            break

after_realtime = scalar("SELECT COUNT(*) FROM news_raw")
print("realtime rows added:", after_realtime - before_realtime)

In [ ]:
realtime_check = rows(
    """
    SELECT symbol, published_at, provider, article_id, publisher, headline
    FROM news_raw
    ORDER BY received_at DESC
    LIMIT 20
    """
)

try:
    import pandas as pd
    display(pd.DataFrame(realtime_check))
except ModuleNotFoundError:
    realtime_check

## 8. 全市场 BroadTape 实时新闻监听测试

这一节不是 watchlist 监听，而是按 `reqNewsProviders()` 返回的 provider 逐个订阅全市场 `NEWS` broad tape。它会尝试 `PROVIDER:PROVIDER_ALL` 形式的 NEWS 合约，并把收到的新闻以 `ALL_NEWS` 写入 `news_raw`。

In [ ]:
def configured_provider_codes():
    return [code.strip() for code in PROVIDER_CODES.split("+") if code.strip()]


def available_provider_codes():
    provider_codes = [row["code"] for row in provider_rows if row.get("code")]
    if not provider_codes:
        provider_codes = configured_provider_codes()
    if BROADTAPE_PROVIDER_LIMIT is None:
        return provider_codes
    return provider_codes[:BROADTAPE_PROVIDER_LIMIT]


def make_broad_tape_contract(provider_code):
    contract = Contract()
    contract.symbol = f"{provider_code}:{provider_code}_ALL"
    contract.secType = "NEWS"
    contract.exchange = provider_code
    return contract


broad_provider_codes = available_provider_codes()
broad_contracts = {
    provider_code: make_broad_tape_contract(provider_code)
    for provider_code in broad_provider_codes
}

print("broad tape provider count:", len(broad_contracts))
for provider_code, contract in broad_contracts.items():
    print(provider_code, contract)

if not broad_contracts:
    raise RuntimeError("没有可用于 BroadTape 监听测试的 news provider。")

In [ ]:
def ingest_broad_tape_tick(provider_hint, tick):
    provider = str(getattr(tick, "providerCode", getattr(tick, "provider", "")) or provider_hint)
    article_id = str(getattr(tick, "articleId", "") or "")
    headline = readable_ib_text(getattr(tick, "headline", "") or "")
    timestamp = str(getattr(tick, "timeStamp", getattr(tick, "time", "")) or "")
    extra_data = readable_ib_text(getattr(tick, "extraData", "") or "")
    if not provider or not article_id or not headline:
        return False
    return service.ingest_tick_news(
        symbol=BROADTAPE_SYMBOL,
        timestamp=timestamp,
        provider=provider,
        article_id=article_id,
        headline=headline,
        ticker_id=None,
        extra_data=extra_data,
    )


broad_tickers = {}
broad_seen = set()
before_broad = scalar("SELECT COUNT(*) FROM news_raw")

for provider_code, contract in broad_contracts.items():
    broad_tickers[provider_code] = ib.reqMktData(
        contract,
        genericTickList=NEWS_GENERIC_TICKS,
        snapshot=False,
        regulatorySnapshot=False,
    )
    print("subscribed broad tape:", provider_code, contract.symbol, NEWS_GENERIC_TICKS)

deadline = time.time() + BROADTAPE_WAIT_SECONDS
while time.time() < deadline:
    ib.sleep(10)
    observed = 0
    inserted = 0
    for provider_code, ticker in broad_tickers.items():
        for tick in iter_news_ticks(ticker):
            provider = str(getattr(tick, "providerCode", getattr(tick, "provider", "")) or provider_code)
            article_id = str(getattr(tick, "articleId", "") or "")
            key = (provider, article_id)
            if key in broad_seen:
                continue
            broad_seen.add(key)
            observed += 1
            inserted += int(ingest_broad_tape_tick(provider_code, tick))
    service.queue.join()
    current = scalar("SELECT COUNT(*) FROM news_raw")
    print(f"broad tape raw rows: {current} (+{current - before_broad}), observed={observed}, inserted={inserted}")
    if current > before_broad:
        break

after_broad = scalar("SELECT COUNT(*) FROM news_raw")
print("broad tape rows added:", after_broad - before_broad)
print("broad tape observed unique ticks:", len(broad_seen))

In [ ]:
broad_summary = rows(
    """
    SELECT symbol, provider, COUNT(*) AS rows, MAX(received_at) AS latest_received_at
    FROM news_raw
    GROUP BY symbol, provider
    ORDER BY rows DESC, provider ASC
    """
)

try:
    import pandas as pd
    display(pd.DataFrame(broad_summary))
except ModuleNotFoundError:
    broad_summary

## 9. 关闭连接

In [ ]:
for symbol, ticker in globals().get("tickers", {}).items():
    try:
        ib.cancelMktData(ticker.contract)
    except Exception as exc:
        print("cancelMktData failed:", symbol, exc)

for provider_code, ticker in globals().get("broad_tickers", {}).items():
    try:
        ib.cancelMktData(ticker.contract)
    except Exception as exc:
        print("cancel broad tape MktData failed:", provider_code, exc)

service.stop()
if ib.isConnected():
    ib.disconnect()
print("stopped")

## 判断结果

- 连接失败：先查 TWS/Gateway API 设置、端口、client id 冲突。
- `reqNewsProviders()` 为空：可能是新闻权限或 IB API provider 查询没有返回，继续看 IBKR error。
- 历史新闻有入库：新闻抓取主链路正常。
- 历史新闻无入库：优先看 `reqHistoricalNews` 的 IBKR error、provider code 和新闻权限。
- 单票实时新闻默认跳过，因为它可能触发普通股票行情权限；需要时把 `RUN_SINGLE_STOCK_REALTIME=True`。
- BroadTape 有入库：你有权限的全市场新闻流监听正常。
- BroadTape 某些 provider 报权限错误：说明该 provider 出现在查询列表里，但当前账户/API 对对应 broad tape 没开实时权限；看具体 provider code 决定是否保留。